In [1]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/GVP/merged_dataset'

Mounted at /content/drive


In [2]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.6 MB/s eta 0:00:00


In [3]:
import os, random, shutil
from pathlib import Path

random.seed(42)

src_images = Path(DATA_DIR) / 'images'
src_labels = Path(DATA_DIR) / 'labels'

out_dir = Path('/content/dataset')
if out_dir.exists():
    shutil.rmtree(out_dir)

for split in ['train', 'val', 'test']:
    (out_dir / split / 'images').mkdir(parents=True, exist_ok=True)
    (out_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

img_files = sorted([f for f in src_images.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
random.shuffle(img_files)

n = len(img_files)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

splits = {
    'train': img_files[:train_end],
    'val': img_files[train_end:val_end],
    'test': img_files[val_end:]
}

def copy_pair(files, split):
    for img_path in files:
        label_path = src_labels / (img_path.stem + '.txt')
        shutil.copy(img_path, out_dir / split / 'images' / img_path.name)
        if label_path.exists():
            shutil.copy(label_path, out_dir / split / 'labels' / label_path.name)
        else:
            print(f"WARNING: missing label for {img_path.name}")

for split, files in splits.items():
    copy_pair(files, split)
    print(f"{split}: {len(files)} images")

train: 1125 images
val: 241 images
test: 242 images


In [11]:
yaml_content = """path: /content/dataset
train: train/images
val: val/images
test: test/images
names:
  0: Animal
  1: Waste
  2: Person
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

path: /content/dataset
train: train/images
val: val/images
test: test/images
names:
  0: Animal
  1: Waste
  2: Person



In [12]:
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project='/content/drive/MyDrive/garbage_runs',
    name='seg_correct_v2'
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=seg_correct_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

In [13]:
metrics = model.val()
print("Box mAP:", metrics.box.map)
print("Seg mAP:", metrics.seg.map)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,977 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1858.1±820.3 MB/s, size: 3086.6 KB)
val: Scanning /content/dataset/val/labels.cache... 241 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 241/241 59.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.2it/s 12.8s
                   all        241       2220      0.778      0.647      0.721      0.357      0.726      0.604      0.654      0.262
                Animal         46        153      0.765      0.706      0.766      0.345       0.68      0.627      0.653      0.221
                 Waste        231       1858      0.757       0.57       0.67      0.328      0.694      0.525      0.591      0.231
                Person         62        

In [14]:
test_metrics = model.val(split='test')
print("Test Box mAP:", test_metrics.box.map)
print("Test Seg mAP:", test_metrics.seg.map)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1046.3±419.8 MB/s, size: 423.7 KB)
val: Scanning /content/dataset/test/labels.cache... 242 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 242/242 72.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.2it/s 13.1s
                   all        242       2064      0.689      0.704      0.704      0.355      0.635      0.647      0.631      0.253
                Animal         44        131      0.738      0.776      0.785      0.348      0.673      0.706      0.705      0.229
                 Waste        239       1782      0.629       0.63      0.639      0.313      0.575      0.573      0.554      0.223
                Person         66        151      0.699      0.706      0.689      0.404      0.658      0.663      0.634      0.3

In [15]:
metrics = model.val()

# Overall (all classes averaged)
print("Box mAP50:", metrics.box.map50)
print("Box mAP50-95:", metrics.box.map)
print("Box Precision:", metrics.box.mp)
print("Box Recall:", metrics.box.mr)

print("Mask mAP50:", metrics.seg.map50)
print("Mask mAP50-95:", metrics.seg.map)
print("Mask Precision:", metrics.seg.mp)
print("Mask Recall:", metrics.seg.mr)

# Per-class breakdown
print("Per-class Box mAP50-95:", metrics.box.maps)  # array indexed by class

# Class names for reference
print(model.names)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1863.9±522.6 MB/s, size: 434.8 KB)
val: Scanning /content/dataset/val/labels.cache... 241 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 241/241 67.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.2it/s 13.0s
                   all        241       2220      0.778      0.647      0.721      0.357      0.726      0.604      0.654      0.262
                Animal         46        153      0.765      0.706      0.766      0.345       0.68      0.627      0.653      0.221
                 Waste        231       1858      0.757       0.57       0.67      0.328      0.694      0.525      0.591      0.231
                Person         62        209      0.812      0.665      0.726        0.4      0.806       0.66      0.719      0.33

In [16]:
import json

summary = {
    "model": "yolo11s-seg",
    "epochs_config": 100,
    "box_map50": metrics.box.map50,
    "box_map50_95": metrics.box.map,
    "box_precision": metrics.box.mp,
    "box_recall": metrics.box.mr,
    "mask_map50": metrics.seg.map50,
    "mask_map50_95": metrics.seg.map,
    "mask_precision": metrics.seg.mp,
    "mask_recall": metrics.seg.mr,
    "per_class_box_map50_95": metrics.box.maps.tolist(),
    "classes": model.names
}

with open('/content/drive/MyDrive/garbage_runs/seg_v1/summary_yolov11.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(summary)

{'model': 'yolo11s-seg', 'epochs_config': 100, 'box_map50': np.float64(0.7208646359673923), 'box_map50_95': np.float64(0.35748350018471364), 'box_precision': np.float64(0.778102575503027), 'box_recall': np.float64(0.6471336337536211), 'mask_map50': np.float64(0.6538939306866964), 'mask_map50_95': np.float64(0.2620545111387782), 'mask_precision': np.float64(0.726447285598817), 'mask_recall': np.float64(0.6041652886074299), 'per_class_box_map50_95': [0.3448040758468668, 0.32798088684208543, 0.39966553786518866], 'classes': {0: 'Animal', 1: 'Waste', 2: 'Person'}}
